# Capacitated Vehicle Routing Problem via Column Generation

## Overview

We solve the CVRP using **column generation**: we iteratively solve a relaxed **Restricted Master Problem (RMP)** and add new feasible vehicle routes (columns) via a **pricing subproblem** until no route with negative reduced cost exists.

The problem has:
- 1 **depot** (node 0) and **24 customer nodes** on a 2-D Euclidean plane
- Each vehicle carries at most **Q = 5** customers (each requiring 1 unit of capacity)
- As many vehicles as needed; goal is to **minimise total distance**

---

### Restricted Master Problem (RMP)

Let $R$ be the current column pool (feasible routes). Each route $r \in R$ has cost $c_r$ and indicator $a_{ir} = 1$ if customer $i$ is in route $r$.

$$
\begin{aligned}
\min \quad & \sum_{r \in R} c_r \,\lambda_r \\
\text{s.t.} \quad & \sum_{r \in R} a_{ir}\,\lambda_r \geq 1 & \forall\,i \in \text{Customers} & \quad (\text{dual: } \pi_i) \\
& \lambda_r \geq 0 & \forall\,r \in R
\end{aligned}
$$

### Pricing Subproblem

Find a feasible vehicle route (depot → customers → depot, $|\text{route}| \leq Q$) with **minimum reduced cost**:

$$\bar{c}_r = c_r - \sum_{i \in r} \pi_i$$

Modelled as a shortest-path MIP with binary arc variables $y_{ij}$, flow-conservation constraints, capacity constraint, and MTZ subtour elimination. If $\bar{c}_r^* < 0$, add the route to $R$ and re-solve the RMP.

### Column Generation Loop

| Step | Action |
|------|--------|
| **Seed** | Start with one singleton route per customer |
| **Solve RMP** | Solve LP relaxation; extract duals $\pi_i$ |
| **Price** | Solve pricing subproblem; find route with min reduced cost $\bar{c}^*$ |
| **Feasible?** | If $\bar{c}^* \geq 0$ → **LP optimal**, stop column generation |
| **Add** | Append new route to $R$; go to Solve |
| **MIP** | Fix column pool; solve integer set-partitioning MIP for final solution |

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pyomo highspy numpy matplotlib networkx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import math, time, warnings, itertools
warnings.filterwarnings('ignore')

import pyomo.environ as pyo

try:
    import highspy
except ImportError:
    raise ImportError("Install highspy:  pip install highspy")

try:
    solver_test = pyo.SolverFactory('appsi_highs')
    print(f"Pyomo appsi_highs available: {solver_test.available()}")
except Exception as e:
    raise ImportError(f"Pyomo appsi_highs not available: {e}")

## 1. Problem Instance

We place **24 customers** and **1 depot** (node 0) uniformly at random in a $[0,100]^2$ square. The depot is fixed at the centre $(50, 50)$. Each customer requires exactly **1 unit** of capacity; each vehicle can carry **Q = 5** customers.

In [ ]:
# ── Problem parameters ────────────────────────────────────────────────────────
DEPOT       = 0
N_CUSTOMERS = 24
Q           = 5          # vehicle capacity (number of customers per vehicle)
N_NODES     = N_CUSTOMERS + 1   # depot + customers

CUSTOMERS = list(range(1, N_NODES))   # node indices 1..24

# ── Random instance ───────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

coords = np.empty((N_NODES, 2))
coords[DEPOT]    = [50.0, 50.0]                          # depot at centre
coords[CUSTOMERS] = rng.uniform(5, 95, size=(N_CUSTOMERS, 2))  # customers

# Euclidean distance matrix (symmetric)
def euclidean_distance_matrix(coords: np.ndarray) -> np.ndarray:
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt((diff ** 2).sum(-1))

D = euclidean_distance_matrix(coords)

# ── Visualise the instance ────────────────────────────────────────────────────
_PALETTE = list(mcolors.TABLEAU_COLORS.values())   # shared colour palette

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(coords[CUSTOMERS, 0], coords[CUSTOMERS, 1],
           s=120, color='steelblue', edgecolors='white', linewidths=1.2,
           zorder=3, label='Customers')
ax.scatter(*coords[DEPOT], s=250, color='crimson', marker='*', zorder=4,
           edgecolors='black', linewidths=0.8, label='Depot (0)')
for i in CUSTOMERS:
    ax.annotate(str(i), coords[i], textcoords='offset points',
                xytext=(5, 4), fontsize=9, fontweight='bold', color='navy')
ax.annotate('0', coords[DEPOT], textcoords='offset points',
            xytext=(6, 4), fontsize=11, fontweight='bold', color='crimson')
ax.set_title(f"CVRP Instance  |  {N_CUSTOMERS} customers  |  Q = {Q}", fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Nodes: {N_NODES}  (depot + {N_CUSTOMERS} customers)   |   Capacity Q = {Q}")

## 2. Route Representation and Initial Columns

A **route** is a list of customer nodes visited in order. The full path starts and ends at the depot:

$$\text{0} \to c_1 \to c_2 \to \cdots \to c_k \to \text{0}, \qquad k \leq Q$$

We seed the column pool with **singleton routes** (one customer each) — these are trivially feasible and guarantee the RMP has an initial feasible solution.

In [ ]:
def route_cost(route: list[int], D: np.ndarray, depot: int = 0) -> float:
    """
    Total travel distance of a route (including depot legs).

    Path: depot -> route[0] -> ... -> route[-1] -> depot
    """
    if not route:
        return 0.0
    cost = D[depot, route[0]]
    for a, b in zip(route, route[1:]):
        cost += D[a, b]
    cost += D[route[-1], depot]
    return cost


def is_feasible(route: list[int], capacity: int) -> bool:
    """A route is feasible iff it visits at most `capacity` customers."""
    return len(route) <= capacity


# ── Seed the column pool with one singleton route per customer ────────────────
initial_routes: list[list[int]] = [[c] for c in CUSTOMERS]
initial_costs:  list[float]     = [route_cost(r, D, DEPOT) for r in initial_routes]

print(f"Initial column pool: {len(initial_routes)} singleton routes")
print(f"Example: route {initial_routes[0]}  cost = {initial_costs[0]:.2f}")
print(f"Min cost singleton: {min(initial_costs):.2f}  |  Max: {max(initial_costs):.2f}")

## 3. Restricted Master Problem (RMP)

The RMP is formulated as an **LP relaxation** of a set-covering problem. We solve it to obtain **dual variables** $\pi_i$ for each coverage constraint, which feed the pricing subproblem.

$$
\begin{aligned}
\min \quad & \sum_{r \in R} c_r \,\lambda_r \\
\text{s.t.} \quad & \sum_{r \in R} a_{ir}\,\lambda_r \geq 1 \qquad \forall\, i \in \text{Customers} \quad (\pi_i \geq 0) \\
& \lambda_r \geq 0 \qquad \forall\,r \in R
\end{aligned}
$$

The dual $\pi_i$ measures how much the objective would improve (per unit) if we relaxed the coverage requirement for customer $i$. These shadow prices guide the pricing subproblem to find routes that are "cheapest once credit for serving customers is applied".

In [ ]:
def build_rmp(
    routes:     list[list[int]],
    costs:      list[float],
    customers:  list[int],
) -> pyo.ConcreteModel:
    """
    Build the LP relaxation of the set-covering Restricted Master Problem.

    Parameters
    ----------
    routes    : list of route definitions (each a list of customer node indices)
    costs     : corresponding route costs
    customers : list of customer node indices that must be covered

    Returns
    -------
    Pyomo ConcreteModel with a `dual` Suffix attached for dual-variable retrieval.
    """
    model = pyo.ConcreteModel(name="VRP_RMP")

    R = list(range(len(routes)))

    # ── Suffix to import dual variables after LP solve ────────────────────────
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

    # ── Decision variables: route selection weights (LP relaxation) ───────────
    model.lam = pyo.Var(R, domain=pyo.NonNegativeReals)

    # ── Objective: minimise total route cost ──────────────────────────────────
    model.obj = pyo.Objective(
        expr=sum(costs[r] * model.lam[r] for r in R),
        sense=pyo.minimize,
    )

    # ── Coverage constraints: each customer covered at least once ─────────────
    # Keyed by customer index so duals are easy to look up
    def cover_rule(m, i):
        return sum(m.lam[r] for r in R if i in routes[r]) >= 1

    model.cover = pyo.Constraint(customers, rule=cover_rule)

    return model


def solve_rmp(model: pyo.ConcreteModel, solver) -> tuple[float, dict[int, float]]:
    """
    Solve the RMP LP and return (objective_value, {customer: dual_value}).

    The dual of each coverage constraint is the shadow price π_i.
    Because the constraint is ≥ 1 (minimisation), duals from HiGHS will be ≤ 0
    internally; we negate them to get π_i ≥ 0 as used in the reduced-cost formula.
    """
    result = solver.solve(model)
    obj_val = pyo.value(model.obj)

    # HiGHS returns duals with sign convention: negative for ≥ constraints in min
    duals = {}
    for i in model.cover:
        raw = model.dual.get(model.cover[i], 0.0)
        # Negate so π_i ≥ 0 (credit for covering customer i)
        duals[i] = -raw if raw is not None else 0.0

    return obj_val, duals

## 4. Pricing Subproblem

The pricing subproblem finds a vehicle route (depot — customers — depot, $\leq Q$ customers) with **minimum reduced cost**, modelled as a MIP over **undirected edges**.

$$\bar{c}_r = \underbrace{\sum_{\{i,j\} \in r} d_{ij}}_{\text{travel cost}} - \underbrace{\sum_{i \in r} \pi_i}_{\text{dual credit}}$$

### Formulation

Let $E_d = \{\{0,j\} : j \in C\}$ (depot–customer edges) and $E_c = \{\{i,j\} : i < j,\; i,j \in C\}$ (customer–customer edges).

**Variables:**
- $x^d_j \in \{0,1,2\}$ — depot–customer edge $\{0,j\}$ multiplicity (value 2 = single-customer out-and-back)
- $x^c_{ij} \in \{0,1\}$ — undirected customer–customer edge $\{i,j\}$, $i < j$
- $v_i \in \{0,1\}$ — 1 if customer $i$ is visited

**Objective:**
$$\min \sum_{j \in C} d_{0j}\,x^d_j + \sum_{\{i,j\}\in E_c} d_{ij}\,x^c_{ij} - \sum_{i \in C} \pi_i\,v_i$$

**Constraints:**
1. $\sum_{j \in C} x^d_j = 2$ — depot has degree 2 (connects to both route endpoints)
2. $x^d_i + \!\!\sum_{\{i,j\}\in E_c}\!\! x^c_{ij} = 2\,v_i \quad \forall\, i \in C$ — customer degree is 0 (unvisited) or 2 (visited)
3. $\sum_{i \in C} v_i \leq Q$ — vehicle capacity
4. $\displaystyle\sum_{\{i,j\}\subseteq S,\,\{i,j\}\in E_c} x^c_{ij} \leq |S|-1 \quad \forall\,S\subseteq C,\;3\leq|S|\leq Q$ — subtour elimination (SECs, prevent isolated customer cycles)

In [ ]:
def build_pricing_subproblem(
    D:         np.ndarray,
    duals:     dict[int, float],
    customers: list[int],
    depot:     int,
    capacity:  int,
) -> pyo.ConcreteModel:
    """
    Build the pricing subproblem MIP with UNDIRECTED edges.

    Finds the vehicle route (depot — customers — depot, at most Q customers)
    with minimum reduced cost using undirected edge variables, degree
    constraints, and subtour-elimination constraints (SECs) for all customer
    subsets of size 3..Q.

    Parameters
    ----------
    D         : N×N symmetric Euclidean distance matrix
    duals     : {customer_index: π_i} dual variables from the current RMP solve
    customers : list of customer node indices
    depot     : depot node index (0)
    capacity  : vehicle capacity Q

    Returns
    -------
    Pyomo ConcreteModel ready to solve.
    """
    model = pyo.ConcreteModel(name="VRP_Pricing")

    # ── Undirected edge sets ──────────────────────────────────────────────────
    # Depot–customer edges: indexed by customer j; allow multiplicity 0/1/2
    # (multiplicity 2 = single-customer out-and-back, e.g. route [depot,j,depot])
    D_EDGES  = list(customers)
    CC_EDGES = [(i, j) for i in customers
                        for j in customers if i < j]     # customer pairs (i < j)

    # ── Decision variables ────────────────────────────────────────────────────
    # x_d[j] ∈ {0,1,2}: multiplicity of undirected depot–customer edge {0,j}
    model.x_d = pyo.Var(D_EDGES,  domain=pyo.NonNegativeIntegers, bounds=(0, 2))
    # x_c[i,j] ∈ {0,1}: undirected customer–customer edge {i,j}, i < j
    model.x_c = pyo.Var(CC_EDGES, domain=pyo.Binary)
    # v[i] ∈ {0,1}: 1 if customer i is visited
    model.v   = pyo.Var(D_EDGES,  domain=pyo.Binary)

    # ── Degree helper ─────────────────────────────────────────────────────────
    def cust_degree(i):
        """Total undirected degree of customer i (depot edge + CC edges)."""
        d = model.x_d[i]
        for (a, b) in CC_EDGES:
            if a == i or b == i:
                d = d + model.x_c[a, b]
        return d

    # ── Objective: minimise reduced cost = travel cost − dual credits ─────────
    model.obj = pyo.Objective(
        expr=(
            sum(D[depot, j] * model.x_d[j] for j in D_EDGES)
            + sum(D[i, j]   * model.x_c[i, j] for (i, j) in CC_EDGES)
            - sum(duals.get(i, 0.0) * model.v[i] for i in D_EDGES)
        ),
        sense=pyo.minimize,
    )

    # ── Depot degree = 2 ──────────────────────────────────────────────────────
    # Depot connects to both endpoints of the route; for a single-stop route
    # the same customer is connected twice (x_d[j] = 2).
    model.depot_degree = pyo.Constraint(
        expr=sum(model.x_d[j] for j in D_EDGES) == 2
    )

    # ── Customer degree: 2 if visited, 0 otherwise ────────────────────────────
    model.cust_degree_con = pyo.ConstraintList()
    for i in D_EDGES:
        model.cust_degree_con.add(cust_degree(i) == 2 * model.v[i])

    # ── Capacity: at most Q customers per route ───────────────────────────────
    model.capacity_con = pyo.Constraint(
        expr=sum(model.v[i] for i in D_EDGES) <= capacity
    )

    # ── Subtour elimination constraints (SECs) ────────────────────────────────
    # For every customer subset S of size 3..Q, the number of edges within S
    # is at most |S|-1, ruling out isolated customer cycles.
    # Enumeration is feasible because Q = 5 is small.
    model.sec = pyo.ConstraintList()
    for size in range(3, capacity + 1):
        for S in itertools.combinations(customers, size):
            S_set = set(S)
            edges_in_S = [(a, b) for (a, b) in CC_EDGES
                          if a in S_set and b in S_set]
            if edges_in_S:
                model.sec.add(
                    sum(model.x_c[a, b] for (a, b) in edges_in_S) <= size - 1
                )

    return model


def extract_route_from_pricing(
    model:     pyo.ConcreteModel,
    depot:     int,
    customers: list[int],
) -> list[int]:
    """
    Reconstruct the ordered customer sequence from undirected edge variables.

    Traces the Hamiltonian path starting from the depot through active edges.
    Returns an ordered list of customer nodes (depot legs not included).
    """
    CC_EDGES = [(i, j) for i in customers for j in customers if i < j]

    # ── Build undirected adjacency for depot + visited customers ──────────────
    adj: dict[int, list[int]] = {depot: []}

    for j in customers:
        val = pyo.value(model.x_d[j])
        if val is None:
            continue
        count = int(round(val))
        for _ in range(count):          # count=2 → depot is adjacent to j twice
            adj[depot].append(j)
            adj.setdefault(j, []).append(depot)

    for (i, j) in CC_EDGES:
        val = pyo.value(model.x_c[i, j])
        if val is not None and val > 0.5:
            adj.setdefault(i, []).append(j)
            adj.setdefault(j, []).append(i)

    if not adj[depot]:
        return []

    # ── Single-customer route: both depot neighbours are the same node ────────
    if len(adj[depot]) >= 2 and adj[depot][0] == adj[depot][1]:
        return [adj[depot][0]]

    # ── Multi-customer route: walk the undirected path from depot ─────────────
    route: list[int] = []
    prev, cur = depot, adj[depot][0]
    while cur != depot:
        route.append(cur)
        nxt = next((nb for nb in adj.get(cur, []) if nb != prev), None)
        if nxt is None:
            break
        prev, cur = cur, nxt

    return route

## 5. Visualisation Helpers

Two plotting functions:
- **`plot_cg_iteration`** — shows the current RMP LP solution (fractional routes with $\lambda > 0$) and highlights the new column entering from the pricing subproblem.
- **`plot_final_solution`** — draws the final integer VRP solution with each vehicle route in a distinct colour.

In [ ]:
def _draw_route(ax, route, depot, coords, color, lw=1.8, alpha=0.75,
                ls='-', label=None, zorder=2):
    """Draw a single vehicle route (depot → customers → depot) as arrows."""
    path = [depot] + route + [depot]
    xs = coords[path, 0]
    ys = coords[path, 1]
    ax.plot(xs, ys, ls=ls, color=color, lw=lw, alpha=alpha,
            zorder=zorder, label=label)


def plot_cg_iteration(
    coords:       np.ndarray,
    routes:       list[list[int]],
    lambdas:      list[float],
    new_route:    list[int] | None,
    new_rc:       float | None,
    iteration:    int,
    rmp_obj:      float,
    depot:        int,
    ax:           plt.Axes | None = None,
) -> None:
    """
    Plot one column generation iteration.

    Active RMP routes (λ > 0.01) are drawn with width proportional to λ.
    The newly added route from pricing is highlighted in red.

    Parameters
    ----------
    ax : if provided, draw into existing Axes; otherwise create & show a new figure.
    """
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 5))

    # ── Plot active RMP routes ────────────────────────────────────────────────
    active_r = [(r, lam) for r, lam in zip(routes, lambdas) if lam > 0.01]
    for idx, (route, lam) in enumerate(active_r):
        color = _PALETTE[idx % len(_PALETTE)]
        lw    = max(0.8, 4.0 * lam)   # width proportional to λ
        _draw_route(ax, route, depot, coords, color=color, lw=lw, alpha=0.7)
        # Mark customers
        ax.scatter(coords[route, 0], coords[route, 1],
                   s=60, color=color, zorder=3,
                   edgecolors='white', linewidths=0.8)

    # ── Highlight new route in bold red ──────────────────────────────────────
    if new_route:
        _draw_route(ax, new_route, depot, coords,
                    color='red', lw=2.5, alpha=0.95, ls='--',
                    label=f'New route  rc={new_rc:.3f}', zorder=5)
        ax.scatter(coords[new_route, 0], coords[new_route, 1],
                   s=100, color='red', zorder=6,
                   edgecolors='black', linewidths=0.8)
        for node in new_route:
            ax.annotate(str(node), coords[node],
                        textcoords='offset points', xytext=(4, 3),
                        fontsize=7, color='darkred', fontweight='bold')
        ax.legend(fontsize=7, loc='upper left', framealpha=0.9)

    # ── Depot marker ─────────────────────────────────────────────────────────
    ax.scatter(*coords[depot], s=200, color='crimson', marker='*',
               zorder=7, edgecolors='black', linewidths=0.8)

    n_active = len(active_r)
    ax.set_title(
        f"Iter {iteration}  |  RMP obj={rmp_obj:.2f}\n"
        f"Active routes (λ>0): {n_active}  |  Pool size: {len(routes)}",
        fontsize=8,
    )
    ax.set_xlabel('x', fontsize=7); ax.set_ylabel('y', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)

    if own_fig:
        plt.tight_layout()
        plt.show()


def plot_final_solution(
    coords:    np.ndarray,
    routes:    list[list[int]],
    lambdas:   list[float],
    obj_val:   float,
    depot:     int,
    elapsed:   float,
) -> None:
    """Draw the final integer VRP solution."""
    fig, ax = plt.subplots(figsize=(8, 7))

    selected = [(r, int(round(lam))) for r, lam in zip(routes, lambdas) if lam > 0.5]
    patches  = []

    # All customer nodes (background scatter)
    ax.scatter(coords[1:, 0], coords[1:, 1], s=80, color='lightgrey',
               zorder=1, edgecolors='white', linewidths=0.8)

    for v_idx, (route, _) in enumerate(selected):
        color = _PALETTE[v_idx % len(_PALETTE)]
        cost  = route_cost(route, D, depot)
        label = f"Vehicle {v_idx+1}: {route}  ({cost:.1f})"
        _draw_route(ax, route, depot, coords, color=color, lw=2.2,
                    alpha=0.85, label=label, zorder=2)
        ax.scatter(coords[route, 0], coords[route, 1],
                   s=120, color=color, zorder=3,
                   edgecolors='white', linewidths=1.0)
        for node in route:
            ax.annotate(str(node), coords[node],
                        textcoords='offset points', xytext=(5, 4),
                        fontsize=8, fontweight='bold', color=color)
        patches.append(mpatches.Patch(color=color, label=label))

    # Depot
    ax.scatter(*coords[depot], s=300, color='crimson', marker='*',
               zorder=8, edgecolors='black', linewidths=1.0)
    ax.annotate('Depot', coords[depot], textcoords='offset points',
                xytext=(7, 5), fontsize=9, color='crimson', fontweight='bold')

    ax.set_title(
        f"Final Integer VRP Solution  |  Total distance = {obj_val:.2f}\n"
        f"Vehicles used: {len(selected)}  |  Wall time: {elapsed:.2f}s",
        fontsize=12,
    )
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(handles=patches, fontsize=7, loc='upper right',
              framealpha=0.9, title='Routes')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 6. Column Generation Loop

We iterate the RMP–Pricing cycle until the pricing subproblem can no longer find a route with negative reduced cost, which certifies LP optimality of the full master problem.

In [ ]:
# ── Solver configuration ──────────────────────────────────────────────────────
lp_solver  = pyo.SolverFactory('appsi_highs')
lp_solver.options['log_to_console'] = False
lp_solver.options['presolve']       = 'on'

mip_solver = pyo.SolverFactory('appsi_highs')
mip_solver.options['log_to_console'] = False
mip_solver.options['mip_rel_gap']    = 1e-6

# ── Initialise column pool ────────────────────────────────────────────────────
routes: list[list[int]]  = [r.copy() for r in initial_routes]
costs:  list[float]      = initial_costs.copy()
history: list[dict]      = []

EPSILON  = 1e-6    # reduced-cost tolerance
MAX_ITER = 150

t_start  = time.perf_counter()

print(f"{'Iter':>5}  {'RMP LP obj':>12}  {'New route':>25}  {'Red. cost':>10}  {'Pool':>5}  {'Time(s)':>8}")
print("─" * 75)

for iteration in range(1, MAX_ITER + 1):

    # ── Solve LP relaxation of RMP ────────────────────────────────────────────
    rmp = build_rmp(routes, costs, CUSTOMERS)
    rmp_obj, duals = solve_rmp(rmp, lp_solver)

    # Extract current λ values
    lambdas = [pyo.value(rmp.lam[r]) for r in range(len(routes))]

    # ── Solve pricing subproblem ──────────────────────────────────────────────
    pricing = build_pricing_subproblem(D, duals, CUSTOMERS, DEPOT, Q)
    mip_solver.solve(pricing)

    reduced_cost = pyo.value(pricing.obj)
    new_route    = extract_route_from_pricing(pricing, DEPOT, CUSTOMERS)

    elapsed = time.perf_counter() - t_start

    # ── Record history ────────────────────────────────────────────────────────
    history.append({
        "iteration":   iteration,
        "rmp_obj":     rmp_obj,
        "duals":       duals.copy(),
        "reduced_cost": reduced_cost,
        "new_route":   new_route.copy() if new_route else [],
        "lambdas":     lambdas.copy(),
        "routes":      [r.copy() for r in routes],   # snapshot before adding
        "pool_size":   len(routes),
    })

    route_str = str(new_route[:6]) + ('...' if len(new_route) > 6 else '')
    print(f"{iteration:>5}  {rmp_obj:>12.4f}  {route_str:>25}  {reduced_cost:>10.4f}"
          f"  {len(routes):>5}  {elapsed:>8.3f}")

    # ── Termination check: no improving column ────────────────────────────────
    if reduced_cost >= -EPSILON:
        print(f"\n✓ Column generation converged after {iteration} iteration(s).")
        print(f"  LP lower bound  : {rmp_obj:.4f}")
        print(f"  Routes in pool  : {len(routes)}")
        print(f"  Wall time       : {elapsed:.3f}s")
        break

    # ── Add new column to pool ────────────────────────────────────────────────
    if new_route and frozenset(new_route) not in [frozenset(r) for r in routes]:
        routes.append(new_route)
        costs.append(route_cost(new_route, D, DEPOT))

else:
    print(f"Warning: reached maximum iterations ({MAX_ITER}) without convergence.")

## 7. Integer Solution via MIP

After column generation converges, we freeze the column pool and solve a **set-partitioning MIP** with binary $\lambda_r \in \{0,1\}$. The coverage constraints are tightened to **equality** (each customer served by exactly one vehicle) to get an integer feasible VRP solution.

In [ ]:
# ── Build final set-partitioning MIP ─────────────────────────────────────────
final_model = pyo.ConcreteModel(name="VRP_Final_MIP")
R_final = list(range(len(routes)))

final_model.lam = pyo.Var(R_final, domain=pyo.Binary)

final_model.obj = pyo.Objective(
    expr=sum(costs[r] * final_model.lam[r] for r in R_final),
    sense=pyo.minimize,
)

# Equality coverage: each customer served exactly once
final_model.cover = pyo.ConstraintList()
for i in CUSTOMERS:
    final_model.cover.add(
        sum(final_model.lam[r] for r in R_final if i in routes[r]) == 1
    )

# ── Solve ─────────────────────────────────────────────────────────────────────
result_mip = mip_solver.solve(final_model)
elapsed_total = time.perf_counter() - t_start

mip_obj  = pyo.value(final_model.obj)
lam_vals = [pyo.value(final_model.lam[r]) for r in R_final]

selected_routes = [routes[r] for r in R_final if lam_vals[r] > 0.5]
n_vehicles      = len(selected_routes)

print(f"MIP Status: {result_mip.solver.termination_condition}")
print(f"\nFinal integer solution:")
print(f"  Total distance : {mip_obj:.4f}")
print(f"  Vehicles used  : {n_vehicles}")
print(f"  Total time     : {elapsed_total:.3f}s")
print(f"\nRoutes selected:")
for v, route in enumerate(selected_routes, 1):
    print(f"  Vehicle {v:2d}:  cost={route_cost(route, D, DEPOT):.2f}")

## 8. Iteration Visualisation

Each panel shows the **RMP LP solution** at that iteration — routes with $\lambda > 0$ are drawn (width ∝ $\lambda$) — plus the **new route** (dashed red) entering from the pricing subproblem.

In [ ]:
# ── Plot all CG iterations in a 3-column grid ─────────────────────────────────
N_COLS  = 3
n_iters = len(history)
n_rows  = math.ceil(n_iters / N_COLS)

fig, axes = plt.subplots(
    n_rows, N_COLS,
    figsize=(6 * N_COLS, 5 * n_rows),
    squeeze=False,
)

for idx, h in enumerate(history):
    row, col = divmod(idx, N_COLS)
    ax = axes[row][col]

    # Show routes & lambdas from this iteration's RMP snapshot
    plot_cg_iteration(
        coords     = coords,
        routes     = h["routes"],
        lambdas    = h["lambdas"],
        new_route  = h["new_route"] if h["reduced_cost"] < -EPSILON else None,
        new_rc     = h["reduced_cost"],
        iteration  = h["iteration"],
        rmp_obj    = h["rmp_obj"],
        depot      = DEPOT,
        ax         = ax,
    )

# Hide unused axes in the last row
leftover = n_iters % N_COLS
for empty_col in range(leftover if leftover else N_COLS, N_COLS):
    axes[n_rows - 1][empty_col].set_visible(False)

plt.suptitle("Column Generation — All Iterations", fontsize=14, y=1.005)
plt.tight_layout()
plt.show()

## 9. Final Solution Visualisation

The optimal integer solution: each vehicle route is coloured distinctly, the depot is marked with a star, and the legend lists the route composition and cost.

In [ ]:
plot_final_solution(
    coords   = coords,
    routes   = routes,
    lambdas  = lam_vals,
    obj_val  = mip_obj,
    depot    = DEPOT,
    elapsed  = elapsed_total,
)

## 10. Convergence Analysis

Three charts summarise how the column generation procedure converged:

1. **RMP LP objective** — decreases monotonically as better routes enter
2. **Reduced cost of the entering column** — approaches 0 from below at termination  
3. **Cumulative column pool size** — grows by one column per iteration

In [ ]:
iters     = [h["iteration"]    for h in history]
rmp_objs  = [h["rmp_obj"]      for h in history]
red_costs = [h["reduced_cost"] for h in history]
pool_sizes = [h["pool_size"] + (1 if h["reduced_cost"] < -EPSILON else 0)
              for h in history]   # pool size after this iteration's column add

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── RMP LP Objective ──────────────────────────────────────────────────────────
axes[0].plot(iters, rmp_objs, 'o-', color='steelblue', lw=2, ms=6)
axes[0].axhline(mip_obj, color='salmon', ls='--', lw=1.5,
                label=f'MIP optimal = {mip_obj:.2f}')
axes[0].set_xlabel('CG Iteration'); axes[0].set_ylabel('LP Objective Value')
axes[0].set_title('RMP LP Objective vs Iteration')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# ── Reduced cost of entering column ───────────────────────────────────────────
axes[1].plot(iters, red_costs, 's-', color='coral', lw=2, ms=6)
axes[1].axhline(0, color='steelblue', ls='--', lw=1.5, label='rc = 0 (stop)')
axes[1].fill_between(iters, red_costs, 0,
                     where=[rc < 0 for rc in red_costs],
                     alpha=0.25, color='coral', label='Improving region')
axes[1].set_xlabel('CG Iteration'); axes[1].set_ylabel('Reduced Cost')
axes[1].set_title('Entering Column Reduced Cost vs Iteration')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# ── Column pool growth ────────────────────────────────────────────────────────
axes[2].step(iters, pool_sizes, where='post', color='mediumpurple', lw=2)
axes[2].fill_between(iters, pool_sizes, step='post',
                     alpha=0.25, color='mediumpurple')
axes[2].scatter(iters, pool_sizes, s=40, color='mediumpurple', zorder=3)
axes[2].set_xlabel('CG Iteration'); axes[2].set_ylabel('Columns in Pool')
axes[2].set_title('Cumulative Column Pool Size vs Iteration')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Column Generation Convergence', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  CG iterations       : {len(history)}")
print(f"  Final LP bound      : {rmp_objs[-1]:.4f}")
print(f"  Integer solution    : {mip_obj:.4f}")
print(f"  Integrality gap     : {(mip_obj - rmp_objs[-1]) / rmp_objs[-1] * 100:.2f}%")
print(f"  Columns generated   : {len(routes) - N_CUSTOMERS}  (beyond initial singletons)")
print(f"  Vehicles deployed   : {n_vehicles}")
print(f"  Customers served    : {sum(len(r) for r in selected_routes)}")